In [ ]:
import mne
from mne.datasets import eegbci

# Seleccionamos un sujeto
subject = 1

# Runs:
# 4, 8, 12 tomados de motor imagery (imagina mover manos vs pies)
runs = [4, 8, 12]

# Descargar datos
files = eegbci.load_data(subject, runs)

print(files)

In [ ]:
# concatenar archivos raw

raws = [mne.io.read_raw_edf(f, preload=True) for f in files]

raw = mne.concatenate_raws(raws)

In [ ]:
# realizar montaje, en este caso estandar (10-20)

# Renombrar canales a estándar
mne.datasets.eegbci.standardize(raw)

# Montaje (posiciones de electrodos)
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

print(f"Datos cargados: {raw.info}")


In [ ]:
raw.plot(n_channels=10, duration=15)

In [ ]:
events, _ = mne.events_from_annotations(raw)

print(events[:10])

In [ ]:
mne.viz.plot_events(events)

In [ ]:
# Crear un diccionario de eventos

event_id = dict(reposo=1, manos=2, pies=3)

In [ ]:
# Filtrado, solamente nos interesan bandas Mu y Beta

raw_filt = raw.filter(8., 12., fir_design='firwin', skip_by_annotation='edge')

In [ ]:
# Referenciacion

raw_filt_ref = raw_filt.set_eeg_reference('average', projection=True)

In [ ]:
raw_filt_ref.plot_sensors(show_names=True)

In [ ]:
raw_filt_ref.plot(n_channels=10, duration=20)

In [ ]:
epochs = mne.Epochs(raw, events, event_id, tmin=-1, tmax=4, 
                    baseline=(None, 0), preload=True)

print(epochs)

In [ ]:
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
import numpy as np

# 1. Extraer datos y etiquetas solo para 'manos' y 'pies'
# El CSP binario es mucho más robusto para empezar
X = epochs['manos', 'pies'].get_data() 
y = epochs['manos', 'pies'].events[:, -1]

# 2. Configurar el CSP
# n_components=4 significa que nos quedaremos con los 4 filtros espaciales más importantes
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)

# 3. Clasificador LDA (Linear Discriminant Analysis)
# Es el estándar en BCI porque es rápido y no requiere mucho cómputo
lda = LinearDiscriminantAnalysis()

# 4. Unir todo en un Pipeline (esto automatiza el proceso)
clf = Pipeline([('CSP', csp), ('LDA', lda)])

# 5. Validación cruzada (Cross-Validation)
# Dividimos los datos en 5 partes para probar la precisión real
scores = cross_val_score(clf, X, y, cv=5)

print("\n--- RESULTADOS DEL MODELO BCI ---")
print(f"Precisión por cada iteración: {scores}")
print(f"Precisión promedio: {np.mean(scores) * 100:.2f}%")


In [ ]:
# Ajustar el CSP a los datos para extraer los patrones
csp.fit(X, y)

# Graficar los patrones espaciales (Topomaps)
csp.plot_patterns(epochs.info, ch_type='eeg', units='Patterns (AU)', size=1.5)


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# 1. Dividir datos en entrenamiento y prueba (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 2. Entrenar el modelo con los datos de entrenamiento
clf.fit(X_train, y_train)

# 3. Predecir los datos que el modelo nunca ha visto
y_pred = clf.predict(X_test)

# 4. Generar y graficar la matriz
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['manos', 'pies'])
disp.plot(cmap=plt.cm.Blues)
plt.title('¿Qué tan bien distingue Manos de Pies?')
plt.show()


In [ ]:
import joblib

# Guardamos el pipeline completo (CSP + LDA)
joblib.dump(clf, 'modelo_bci_entrenado.pkl')
print("Modelo guardado como 'modelo_bci_entrenado.pkl'")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Generamos la matriz con los datos de prueba que ya tenías
y_pred = clf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Manos', 'Pies'])

fig_cm, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix: Motor Imagery')
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()


In [ ]:
# Tomamos un canal (ej: el canal 10 que suele estar cerca de C3)
canal_id = 10 
datos_crudos = raws[0].get_data(picks=[canal_id], tmin=0, tmax=2)[0]
datos_filtrados = raw.get_data(picks=[canal_id], tmin=0, tmax=2)[0]

fig_signal, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(datos_crudos, color='red', alpha=0.5)
ax[0].set_title('Raw Signal (with noise)')
ax[1].plot(datos_filtrados, color='blue')
ax[1].set_title('Filtered Signal (8-14 Hz - Mu Rhythm)')

plt.tight_layout()
plt.savefig('eeg_signal_filter.png', dpi=300)
plt.close()


In [ ]:
import matplotlib.pyplot as plt

# Generamos los patrones del CSP
fig_topo = csp.plot_patterns(epochs.info, ch_type='eeg', show=False)
# Guardamos solo el primer patrón (el que suele mostrar C3)
plt.savefig('topomap_c3.png', dpi=300, bbox_inches='tight')
plt.close()
